In [84]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [85]:
spark = (
    SparkSession.builder.master("local[*]").appName("taxi-revenue-report").getOrCreate()
)
spark.version

'4.1.1'

In [86]:
from common.config import get_root_path

# Declare dataset paths
DATA_PATH = get_root_path() / "data"
TAXI_PATH = DATA_PATH / "taxi"
DATASET_GREEN_PATH = TAXI_PATH / "clean" / "green" / "2025" / "11"
DATASET_YELLOW_PATH = TAXI_PATH / "clean" / "yellow" / "2025" / "11"
DATASET_ZONES = DATA_PATH / "taxi_zone_lookup.csv"
DATASET_REPORT_PATH = TAXI_PATH / "report" / "revenue"
REPORT_REVENUE_ZONE = DATASET_REPORT_PATH / "revenue_by_zone"  # total revenue per zone
REPORT_REVENUE_VENDOR_ZONE = (
    DATASET_REPORT_PATH / "revenue_by_vendor_zone"
)  # total revenue per vendor and zone

In [87]:
print("[INFO] Loading dataset clean/green/2025/11")
green_df = spark.read.parquet(str(DATASET_GREEN_PATH))

[INFO] Loading dataset clean/green/2025/11


In [88]:
green_df = green_df.select("vendor_id", "pickup_location_id", "total_amount").withColumnRenamed("pickup_location_id", "location_id")
green_df.columns

['vendor_id', 'location_id', 'total_amount']

In [89]:
print("[INFO] Loading dataset clean/yellow/2025/11")
yellow_df = spark.read.parquet(str(DATASET_YELLOW_PATH))

[INFO] Loading dataset clean/yellow/2025/11


In [90]:
yellow_df = yellow_df.select("vendor_id", "pickup_location_id", "total_amount").withColumnRenamed("pickup_location_id", "location_id")
yellow_df.columns

['vendor_id', 'location_id', 'total_amount']

In [91]:
df_combined = green_df.union(yellow_df)
df_combined.columns

['vendor_id', 'location_id', 'total_amount']

In [92]:
print("[INFO] Loading dataset taxi_zone_lookup")
df_zones = (
    spark.read.csv(str(DATASET_ZONES), header=True)
    .withColumnsRenamed({"LocationId": "location_id", "Zone": "zone"})
    .select("location_id", "zone")
)

[INFO] Loading dataset taxi_zone_lookup


In [93]:
print("[INFO] Joining taxi with zones")
# broadcast: df_zones is small enough to copy it to all executors, for the later join with taxi dataset
df_zones = F.broadcast(df_zones)
df_joined = df_combined.join(df_zones, on="location_id")

# create temporary table
df_joined.registerTempTable("taxi_with_zones")

[INFO] Joining taxi with zones


In [94]:
# SQL mode version of spark code. For check and debug.
# df_revenue_zone = spark.sql(
# """
# SELECT
#     zone,
#     CAST(SUM(total_amount) AS DECIMAL(20, 2)) as revenue
# FROM taxi_with_zones
# GROUP BY zone
# ORDER BY revenue DESC
# """
# )
# df_revenue_zone.show()
# df_revenue_zone.write.parquet(str(REPORT_REVENUE_ZONE), mode="overwrite")

In [ ]:
print("[INFO] Generating report: total revenue by zone")
df_revenue_zone = (
    df_joined.groupBy("zone")
    .agg(F.sum("total_amount").cast("decimal(20, 2)").alias("revenue"))
    .sort(F.desc("revenue"))
)

[INFO] Generating report: total revenue by zone


In [96]:
# SQL mode version of spark code. For check and debug.
# df_revenue_vendor_zone = spark.sql(
# """
# SELECT
#     vendor_id,
#     zone,
#     CAST(SUM(total_amount) AS DECIMAL(20, 2)) as revenue
# FROM taxi_with_zones
# GROUP BY vendor_id, zone
# ORDER by vendor_id, revenue DESC
# """
# )
# df_revenue_vendor_zone.show()
# df_revenue_vendor_zone.write.parquet(str(REPORT_REVENUE_VENDOR_ZONE), mode="overwrite")

In [97]:
print("[INFO] Generating report: total revenue by vendor_id and zone")
df_revenue_vendor_zone = (
    df_joined.groupBy("vendor_id", "zone")
    .agg(F.sum("total_amount").cast("decimal(20, 2)").alias("revenue"))
    .sort([F.col("vendor_id"), F.desc("revenue")])
)

[INFO] Generating report: total revenue by vendor_id and zone


In [98]:
print("[INFO] Writing report to file: revenue by zon")
df_revenue_zone.write.parquet(str(REPORT_REVENUE_ZONE), mode="overwrite")
print("[INFO] Writing report to file: revenue by vendor_id and zone")
df_revenue_vendor_zone.write.parquet(str(REPORT_REVENUE_VENDOR_ZONE), mode="overwrite")

[INFO] Writing report to file: revenue by zon
[INFO] Writing report to file: revenue by vendor_id and zone


In [99]:
# spark.stop()